This serves as a data cleaning notebook to compile all of the data into necessary state for suitable use.

In [3]:
import pandas as pd
import numpy as np

In [29]:
boxofficemojo_data1to200  = pd.read_html("https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?ref_=bo_lnav_hm_shrt")
boxofficemojo_data200to400 = pd.read_html("https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?offset=200")
boxofficemojo_data401to600 = pd.read_html("https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?offset=400")
boxofficemojo_data601to800 = pd.read_html("https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?offset=600")
boxofficemojo_data801to1000 = pd.read_html("https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/?offset=800")

In [32]:
df1to200 = pd.DataFrame(boxofficemojo_data1to200[0])
df200to400 = pd.DataFrame(boxofficemojo_data200to400[0])
df401to600 = pd.DataFrame(boxofficemojo_data401to600[0])
df601to800 = pd.DataFrame(boxofficemojo_data601to800[0])
df801to1000 = pd.DataFrame(boxofficemojo_data801to1000[0])


df1 = pd.concat([df1to200, df200to400, df401to600, df601to800, df801to1000], ignore_index=True)

In [33]:
df1 = df1[['Rank', 'Title', 'Domestic Lifetime Gross', 'Year']]

In [34]:
df1

,Rank,Title,Domestic Lifetime Gross,Year
0,1,Avengers: Endgame,"$898,471,864",2019
1,2,Avatar,"$785,221,649",2009
2,3,Spider-Man: Brand New Day,"$958,944,095",2026
3,4,Avatar: The Way of Water,"$688,459,501",2022
4,5,Ne Zha 2,"$23,308,176",2025
...,...,...,...,...
995,996,Open Season,"$85,105,259",2006
996,997,Cats & Dogs,"$93,385,515",2001
997,998,The Hunt for Red October,"$122,012,643",1990
998,999,Scare Out,-,2026


In [36]:
import requests
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from getpass import getpass

TOKEN = getpass("Enter your TMDB API Read Access Token: ")

BASE_URL = "https://api.themoviedb.org/3"

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "accept": "application/json"
})

def tmdb_get(endpoint, params=None):
    response = session.get(
        f"{BASE_URL}/{endpoint}",
        params=params,
        timeout=30
    )
    response.raise_for_status()
    return response.json()

c:\Users\Maseg\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
movie = tmdb_get("movie/19995")

print(movie["title"])
print(movie["release_date"])
print(movie["budget"])
print(movie["revenue"])

Avatar
2009-12-16
237000000
2923706026


In [38]:
movies = []

for page in tqdm(range(1, 51)):

    data = tmdb_get(
        "discover/movie",
        params={
            "sort_by": "revenue.desc",
            "include_adult": "false",
            "primary_release_date.lte": "2025-12-31",
            "page": page
        }
    )

    movies.extend(data["results"])

movies = movies[:1000]

print("Movies collected:", len(movies))

100%|██████████| 50/50 [00:02<00:00, 17.21it/s]

Movies collected: 1000


In [39]:
def get_movie_details(movie_id):

    movie = tmdb_get(
        f"movie/{movie_id}",
        params={"append_to_response": "credits"}
    )

    credits = movie.get("credits", {})

    # Director
    directors = [
        person["name"]
        for person in credits.get("crew", [])
        if person.get("job") == "Director"
    ]

    # Top 3 billed actors
    cast = sorted(
        credits.get("cast", []),
        key=lambda x: x.get("order", 999)
    )

    lead_actors = [
        person["name"]
        for person in cast[:3]
    ]

    # Genres
    genres = [
        genre["name"]
        for genre in movie.get("genres", [])
    ]

    # Collection membership is a franchise proxy
    collection = movie.get("belongs_to_collection")

    return {
        "movie_id": movie["id"],
        "title": movie["title"],
        "release_date": movie.get("release_date"),
        "production_budget": movie.get("budget") or np.nan,
        "genre": genres[0] if genres else None,
        "franchise_status": (
            "Franchise" if collection else "Standalone"
        ),
        "director": ", ".join(directors),
        "lead_actors": ", ".join(lead_actors),
        "tmdb_worldwide_revenue": movie.get("revenue") or np.nan
    }

In [40]:
detailed_movies = []
failed_ids = []

for movie in tqdm(movies):

    try:
        details = get_movie_details(movie["id"])
        detailed_movies.append(details)

    except requests.RequestException as e:
        failed_ids.append(movie["id"])
        print(f"Failed: {movie['id']} - {e}")

df = pd.DataFrame(detailed_movies)

print(df.shape)
df.head()

100%|██████████| 1000/1000 [00:52<00:00, 19.20it/s]

(1000, 9)


,movie_id,title,release_date,production_budget,genre,franchise_status,director,lead_actors,tmdb_worldwide_revenue
0,299534,Avengers: Endgame,2019-04-24,356000000.0,Adventure,Franchise,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo",2942186933
1,19995,Avatar,2009-12-16,237000000.0,Science Fiction,Franchise,James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver",2923706026
2,76600,Avatar: The Way of Water,2022-12-14,460000000.0,Action,Franchise,James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver",2334484620
3,980477,Ne Zha 2,2025-01-29,80000000.0,Animation,Franchise,Jiao Zi,"Lu Yanting, Joseph, Han Mo",2270700370
4,597,Titanic,1997-12-18,200000000.0,Drama,Standalone,James Cameron,"Leonardo DiCaprio, Kate Winslet, Billy Zane",2264162353


In [41]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

df["release_year"] = df["release_date"].dt.year

df["release_decade"] = (
    df["release_year"] // 10 * 10
).astype("Int64")

def get_season(month):
    if pd.isna(month):
        return None
    if month in [12, 1, 2]:
        return "Winter"
    if month in [3, 4, 5]:
        return "Spring"
    if month in [6, 7, 8]:
        return "Summer"
    return "Fall"

df["release_season"] = (
    df["release_date"].dt.month.apply(get_season)
)

In [42]:
df.to_csv("tmdb_top_1000_movies.csv", index=False)

print("Dataset exported successfully!")
print(df.shape)
print(df.isna().sum())

Dataset exported successfully!
(1000, 12)
movie_id                   0
title                      0
release_date               0
production_budget         33
genre                      0
franchise_status           0
director                   0
lead_actors                0
tmdb_worldwide_revenue     0
release_year               0
release_decade             0
release_season             0
dtype: int64


In [43]:
df2 = pd.read_csv("tmdb_top_1000_movies.csv")

In [44]:
merged_df = pd.merge(
    df1, df2,
    left_on=["Title", "Year"], right_on=["title", "release_year"],
    how="inner"
)

In [45]:
merged_df

,Rank,Title,Domestic Lifetime Gross,Year,movie_id,title,release_date,production_budget,genre,franchise_status,director,lead_actors,tmdb_worldwide_revenue,release_year,release_decade,release_season
0,1,Avengers: Endgame,"$898,471,864",2019,299534,Avengers: Endgame,2019-04-24,356000000.0,Adventure,Franchise,"Anthony Russo, Joe Russo","Robert Downey Jr., Chris Evans, Mark Ruffalo",2942186933,2019,2010,Spring
1,2,Avatar,"$785,221,649",2009,19995,Avatar,2009-12-16,237000000.0,Science Fiction,Franchise,James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver",2923706026,2009,2000,Winter
2,4,Avatar: The Way of Water,"$688,459,501",2022,76600,Avatar: The Way of Water,2022-12-14,460000000.0,Action,Franchise,James Cameron,"Sam Worthington, Zoe Saldaña, Sigourney Weaver",2334484620,2022,2020,Winter
3,5,Ne Zha 2,"$23,308,176",2025,980477,Ne Zha 2,2025-01-29,80000000.0,Animation,Franchise,Jiao Zi,"Lu Yanting, Joseph, Han Mo",2270700370,2025,2020,Winter
4,6,Titanic,"$674,354,882",1997,597,Titanic,1997-12-18,200000000.0,Drama,Standalone,James Cameron,"Leonardo DiCaprio, Kate Winslet, Billy Zane",2264162353,1997,1990,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
909,995,Valkyrie,"$83,077,833",2008,2253,Valkyrie,2008-12-25,75000000.0,Drama,Standalone,Bryan Singer,"Tom Cruise, Kenneth Branagh, Bill Nighy",200276000,2008,2000,Winter
910,996,Open Season,"$85,105,259",2006,7484,Open Season,2006-09-27,85000000.0,Animation,Franchise,"Roger Allers, Jill Culton","Martin Lawrence, Ashton Kutcher, Gary Sinise",200811689,2006,2000,Fall
911,997,Cats & Dogs,"$93,385,515",2001,10992,Cats & Dogs,2001-07-04,60000000.0,Family,Franchise,Lawrence Guterman,"Jeff Goldblum, Elizabeth Perkins, Alexander Po...",200687492,2001,2000,Summer
912,998,The Hunt for Red October,"$122,012,643",1990,1669,The Hunt for Red October,1990-03-02,30000000.0,Action,Franchise,John McTiernan,"Alec Baldwin, Sean Connery, Scott Glenn",200512643,1990,1990,Spring


In [46]:
merged_df.to_csv("merged_boxoffice_tmdb.csv", index=False)

At this time the datasets are mostly completed, some columns still need to be removed and auxiliary one added, based on calculation/considerations from the dataset.